<a href="https://colab.research.google.com/github/kimjiwoo2/Pill-agent/blob/develop/notebooks/yoonsoo/crop_v5_test%EC%9A%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip uninstall -y torch torchvision torchaudio modelscope
!pip install paddlepaddle-gpu==3.1.0 -i \
    https://www.paddlepaddle.org.cn/packages/stable/cu118/
!pip install "paddleocr>=3.0.0"
!pip install -q opencv-python-headless pandas numpy matplotlib tqdm

In [ ]:
from __future__ import annotations

import re
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

## 1. 경로 설정

In [ ]:
DRIVE_BASE = Path("/content/drive/MyDrive/Pillot/dataset/pilliot_test_set")

# test 입력
TEST_ZIP      = DRIVE_BASE / "crop_v3_test.zip"
TEST_MANIFEST = DRIVE_BASE / "crop_v3_manifest.csv"

# 압축 해제 경로
TEST_DIR = Path("/content/crops_test")

# crop_v5 출력
DRIVE_OUT = DRIVE_BASE
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

print(f"TEST_ZIP      : {TEST_ZIP.exists()}")
print(f"TEST_MANIFEST : {TEST_MANIFEST.exists()}")

## 2. test 압축 해제

In [ ]:
TEST_DIR.mkdir(parents=True, exist_ok=True)

print(f"압축 해제 중: {TEST_ZIP.name} ...")
with zipfile.ZipFile(TEST_ZIP, "r") as zf:
    zf.extractall(TEST_DIR)

n = sum(1 for _ in TEST_DIR.rglob("*.png"))
print(f"완료: {n}개 이미지")

## 3. Manifest 로드

In [ ]:
IGNORE_IMPRINT_TOKENS = {
    "", "NAN", "NONE", "NULL", "마크", "분할선", "없음", "무", "-",
}
_RE_STRIP_TOKENS = re.compile(r"\s+|분할선|마크|\|")
_RE_ALLOWED      = re.compile(r"[^0-9A-Z가-힣+\-/]")

def normalize_imprint(text: object) -> str:
    if pd.isna(text):
        return ""
    text = str(text).strip().upper()
    if text in IGNORE_IMPRINT_TOKENS:
        return ""
    text = _RE_STRIP_TOKENS.sub("", text)
    text = _RE_ALLOWED.sub("", text)
    if text in IGNORE_IMPRINT_TOKENS:
        return ""
    return text

In [ ]:
df = pd.read_csv(TEST_MANIFEST, low_memory=False)

# crop_path를 TEST_DIR 기준으로 재매핑
df["crop_path"] = df["object_id"].apply(lambda oid: str(TEST_DIR / f"{oid}.png"))
df["_crop_exists"] = df["crop_path"].apply(lambda p: Path(p).exists())

# split 컬럼이 없으면 전부 test로 취급
if "split" not in df.columns:
    df["split"] = "test"

print(f"manifest 로드: {df.shape}")
print(f"split 분포:\n{df['split'].value_counts()}")
print(f"crop 파일 존재: {df['_crop_exists'].sum()} / {len(df)}")

df = df[df["_crop_exists"]].reset_index(drop=True)

## 4. 필터링 대상 식별

**필터링 기준**: `drug_dir`가 가리키는 면에 각인이 없고, 반대 면에는 각인이 있는 경우
- `drug_dir == 앞면` AND `print_front` 각인 없음 AND `print_back` 각인 있음
- `drug_dir == 뒷면` AND `print_back` 각인 없음 AND `print_front` 각인 있음

→ 단면 각인 약인데 각인 없는 면이 crop된 케이스를 제거

In [ ]:
df["_front_norm"] = df["print_front"].apply(normalize_imprint)
df["_back_norm"]  = df["print_back"].apply(normalize_imprint)

mask_exclude = (
    ((df["drug_dir"] == "앞면") & (df["_front_norm"] == "") & (df["_back_norm"] != "")) |
    ((df["drug_dir"] == "뒷면") & (df["_back_norm"]  == "") & (df["_front_norm"] != ""))
)

df_exclude = df[mask_exclude].reset_index(drop=True)
df_keep    = df[~mask_exclude].reset_index(drop=True)

print("=" * 40)
print(f"전체       : {len(df):,}건")
print(f"필터링 대상 : {len(df_exclude):,}건 ({len(df_exclude)/len(df)*100:.1f}%)")
print(f"필터링 후   : {len(df_keep):,}건")
print("=" * 40)
print(f"\n[필터링 대상 split 분포]")
print(df_exclude["split"].value_counts())
print(f"\n[필터링 후 split 분포]")
print(df_keep["split"].value_counts())

## 5. 필터링 대상 시각화

In [ ]:
# 샘플 20장 확인
COLS = 4
n    = min(len(df_exclude), 20)
rows = (n + COLS - 1) // COLS

fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 3, rows * 3))
axes = np.array(axes).flatten()

for ax, (_, row) in zip(axes, df_exclude.sample(n, random_state=42).iterrows()):
    img = cv2.imread(row["crop_path"])
    if img is not None:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f'drug_dir: {row["drug_dir"]}\n'
        f'F: {row["print_front"]} / B: {row["print_back"]}',
        fontsize=6
    )
    ax.axis("off")

for ax in axes[n:]:
    ax.axis("off")

plt.suptitle(f"필터링 대상 샘플 (전체 {len(df_exclude)}건 중 {n}건)", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# 전체 필터링 대상 시각화
COLS = 6
n    = len(df_exclude)
rows = (n + COLS - 1) // COLS

fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 3, rows * 2))
axes = np.array(axes).flatten()

for ax, (_, row) in zip(axes, df_exclude.iterrows()):
    img = cv2.imread(row["crop_path"])
    if img is not None:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f'{row["drug_dir"]}\nF:{row["print_front"]} B:{row["print_back"]}',
        fontsize=5
    )
    ax.axis("off")

for ax in axes[n:]:
    ax.axis("off")

plt.suptitle(f"필터링 대상 전체 {n}건", fontsize=10)
plt.tight_layout()
plt.show()

## 5-1. 필터링 대상에 PaddleOCR 실행 → 오라벨 복원 후보 추출

**로직**
- `df_exclude`(각인 없는 면으로 판별된 케이스)에 OCR 실행
- OCR 결과가 있는 것 = `drug_dir` 오라벨 가능성 → 복원 후보로 추출
- 시각화 후 직접 확인하여 복원 여부 결정

In [ ]:
import numpy as np
from paddleocr import PaddleOCR

if "_ocr" not in globals():
    _ocr = PaddleOCR(
        device="gpu",
        use_doc_orientation_classify=True,
        use_doc_unwarping=False,
        use_textline_orientation=True,
        text_detection_model_name="PP-OCRv5_server_det",
        text_recognition_model_name="PP-OCRv5_server_rec",
    )
    print("PaddleOCR 초기화 완료")
else:
    print("PaddleOCR 이미 초기화됨 — 재사용")

In [ ]:
# df_exclude에 OCR 실행
ocr_records = []
for _, row in tqdm(df_exclude.iterrows(), total=len(df_exclude), desc="OCR on excluded"):
    img = cv2.imread(row["crop_path"])
    ocr_text, ocr_raw, ocr_conf = "", "", float("nan")
    if img is not None:
        try:
            result = _ocr.predict(img, text_det_thresh=0.3)
            if result:
                page  = result[0]
                texts = [str(t) for t in page.get("rec_texts", [])]
                confs = [float(c) for c in page.get("rec_scores", [])]
                polys = page.get("rec_polys", [])
                if texts:
                    if polys and len(polys) == len(texts):
                        def _cy(poly): return sum(p[1] for p in poly) / len(poly)
                        def _cx(poly): return sum(p[0] for p in poly) / len(poly)
                        heights = [max(p[1] for p in poly) - min(p[1] for p in poly) for poly in polys]
                        row_thresh = (sum(heights) / len(heights)) / 2
                        items = sorted(
                            zip(texts, polys, confs),
                            key=lambda x: (round(_cy(x[1]) / row_thresh), _cx(x[1]))
                        )
                        texts = [t for t, _, _ in items]
                        confs = [c for _, _, c in items]
                    ocr_raw  = " | ".join(texts)
                    ocr_conf = float(np.mean(confs))
                    ocr_text = normalize_imprint("".join(texts).strip())
        except Exception:
            pass
    ocr_records.append({"object_id": str(row["object_id"]),
                        "ocr_raw": ocr_raw, "ocr_text": ocr_text, "ocr_conf": ocr_conf})

df_exclude_ocr = df_exclude.merge(pd.DataFrame(ocr_records), on="object_id", how="left")
print(f"OCR 완료: {len(df_exclude_ocr)}건")
print(f"OCR 결과 있음: {df_exclude_ocr['ocr_text'].ne('').sum()}건")

In [ ]:
# 복원 후보 추출
df_restore_candidates = df_exclude_ocr[
    df_exclude_ocr["ocr_text"].ne("")
].reset_index(drop=True)

print(f"복원 후보: {len(df_restore_candidates)}건")
display(df_restore_candidates[[
    "object_id", "drug_dir", "print_front", "print_back", "ocr_raw", "ocr_text", "ocr_conf"
]])

In [ ]:
# 복원 후보 전체 시각화 — 직접 보고 복원 여부 판단
COLS = 4
n    = len(df_restore_candidates)

if n == 0:
    print("복원 후보 없음")
else:
    rows = (n + COLS - 1) // COLS
    fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 3, rows * 3))
    axes = np.array(axes).flatten()

    for ax, (idx, row) in zip(axes, df_restore_candidates.iterrows()):
        img = cv2.imread(row["crop_path"])
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(
            f'[{idx}] {row["drug_dir"]}\n'
            f'F: {row["print_front"]} / B: {row["print_back"]}\n'
            f'OCR: {row["ocr_text"]} (conf={row["ocr_conf"]:.2f})',
            fontsize=6
        )
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.suptitle(f"복원 후보 {n}건 — 타이틀의 [index]로 복원 여부 지정", fontsize=10)
    plt.tight_layout()
    plt.show()

### 복원 대상 지정

위 시각화에서 실제로 각인이 있는(= 복원해야 할) 케이스의 **[index] 번호**를 아래 리스트에 입력하세요.

In [ ]:
# 복원 후보 전체를 복원 대상으로 확정
df_restore = df_restore_candidates.reset_index(drop=True)
print(f"복원 대상: {len(df_restore)}건")
display(df_restore[["object_id", "drug_dir", "print_front", "print_back", "ocr_text"]])

In [ ]:
# 복원 대상을 df_keep에 추가하여 최종 데이터셋 구성
restore_ids = set(df_restore["object_id"].astype(str))

df_keep_final = pd.concat(
    [df_keep, df_exclude[df_exclude["object_id"].astype(str).isin(restore_ids)]],
    ignore_index=True
)

print("=" * 40)
print(f"필터링 전   : {len(df):,}건")
print(f"필터링 후   : {len(df_keep):,}건")
print(f"복원 후 최종: {len(df_keep_final):,}건  (+{len(df_restore)}건 복원)")
print("=" * 40)
print(f"\n최종 split 분포:")
print(df_keep_final["split"].value_counts())

## 6. crop_v5 저장 (test zip + manifest)

In [ ]:
zip_path = DRIVE_OUT / "crop_v5_test.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as zf:
    for _, row in tqdm(df_keep_final.iterrows(), total=len(df_keep_final), desc="zip test"):
        src = Path(row["crop_path"])
        if src.exists():
            zf.write(src, arcname=src.name)
print(f"test: {len(df_keep_final)}건  →  {zip_path}  ({zip_path.stat().st_size / 1e9:.2f} GB)")

In [ ]:
# manifest 저장 (내부 임시 컬럼 제거)
df_keep_final.drop(columns=["_crop_exists", "_front_norm", "_back_norm"], errors="ignore") \
       .to_csv(DRIVE_OUT / "crop_v5_test_manifest.csv", index=False)

print("crop_v5_test_manifest.csv 저장 완료")
print(f"최종 건수: {len(df_keep_final):,}건")

In [ ]:
# df_exclude 중 복원 대상으로 확정되지 않은 케이스 (= OCR도 못 찾은 케이스)
restore_ids_check = set(df_restore["object_id"].astype(str))
df_not_restored = df_exclude_ocr[
    ~df_exclude_ocr["object_id"].astype(str).isin(restore_ids_check)
].reset_index(drop=True)

COLS = 6
n    = len(df_not_restored)
rows = (n + COLS - 1) // COLS

if n == 0:
    print("복원되지 않은 케이스 없음")
else:
    fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 3, rows * 2))
    axes = np.array(axes).flatten()

    for ax, (_, row) in zip(axes, df_not_restored.iterrows()):
        img = cv2.imread(row["crop_path"])
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(
            f'{row["drug_dir"]}\nF:{row["print_front"]} B:{row["print_back"]}',
            fontsize=5
        )
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.suptitle(f"복원되지 않은 케이스 전체 {n}건", fontsize=10)
    plt.tight_layout()
    plt.show()